# Q-GEPO: Quantum Tic-Tac-Toe — Green Grid Edition

A two-player Colab game based on Track 1, Idea 2. Solar ⚡ and Wind 🌿 players place each quantum mark across two empty squares. Once the links form a cycle, Qiskit measures the entangled moves and collapses them onto the board. Three in a row wins.

Run the code cell. Enter two different empty square numbers (1–9) when prompted. Use `q` to quit. The Qiskit simulator runs locally; no IBM Quantum account or token is needed.

In [ ]:
%pip -q install "qiskit>=2.0,<3.0"

from collections import defaultdict, deque
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

PLAYERS = ["Solar ⚡", "Wind 🌿"]
WIN_LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def show_board(board):
    cells = [str(i+1) if x is None else x for i,x in enumerate(board)]
    print("\n" + "\n---+---+---\n".join(" | ".join(cells[i:i+3]) for i in (0,3,6)) + "\n")

def winner_of(board):
    for line in WIN_LINES:
        if board[line[0]] is not None and all(board[i] == board[line[0]] for i in line):
            return board[line[0]]
    return None

def closes_cycle(edges, edge):
    graph = defaultdict(list)
    for a,b in edges:
        graph[a].append(b); graph[b].append(a)
    start,target = edge
    queue, seen = deque([start]), {start}
    while queue:
        node = queue.popleft()
        if node == target: return True
        for nxt in graph[node]:
            if nxt not in seen: seen.add(nxt); queue.append(nxt)
    return False

def quantum_collapse(moves):
    # One qubit encodes each unresolved move's choice between its two squares.
    # Prepare Bell pairs so neighboring unresolved move choices are entangled.
    qc = QuantumCircuit(len(moves))
    for q in range(0, len(moves)-1, 2):
        qc.h(q)
        qc.cx(q, q+1)
    if len(moves) % 2: qc.h(len(moves)-1)  # unpaired move remains in superposition
    qc.measure_all()
    counts = StatevectorSampler().run([qc], shots=1).result()[0].data.meas.get_counts()
    bits = next(iter(counts)).replace(" ", "")[::-1]
    return qc, [int(b) for b in bits]

board = [None]*9
moves, edges = [], []
turn, winner = 0, None
print("Q-GEPO: Quantum Tic-Tac-Toe — Green Grid Edition")
print("Choose two different empty squares (1–9). Enter q to quit.")
show_board(board)
while winner is None and any(x is None for x in board):
    player = PLAYERS[turn % 2]
    entry = input(f"{player}, choose two squares: ").strip().lower()
    if entry == "q":
        print("Game ended."); break
    try:
        a,b = [int(x)-1 for x in entry.replace(",", " ").split()]
        if a == b or any(i not in range(9) or board[i] is not None for i in (a,b)): raise ValueError
    except ValueError:
        print("Enter two different empty square numbers from 1 to 9."); continue
    edge = (a,b)
    cycle = closes_cycle(edges, edge)
    edges.append(edge)
    moves.append({"player":player, "cells":edge})
    turn += 1
    print(f"{player} placed a superposed mark across squares {a+1} and {b+1}.")
    if cycle:
        print("Cycle detected: entangled marks collapse!")
        qc, bits = quantum_collapse(moves)
        print(qc.draw(output="text"))
        for move, bit in zip(moves, bits):
            square = move["cells"][bit]
            if board[square] is None:
                board[square] = move["player"]
                print(f"{move['player']} resolves to square {square+1}.")
            else:
                print(f"{move['player']}'s mark collapses onto occupied square {square+1} and is lost.")
        moves.clear(); edges.clear()
        show_board(board)
        winner = winner_of(board)
        if winner: print(f"🏆 {winner} wins the green grid!")
if winner is None and not any(x is None for x in board): print("The grid is full — draw!")
